# VirtualiZarr: NOAA CDR NDVI on AWS S3 → Icechunk on Cloudflare R2

Applies the VirtualiZarr workflow to the **NOAA Climate Data Record (CDR) of AVHRR NDVI** — a 0.05° global daily vegetation index dataset publicly available on AWS S3. Virtual references are stored in an Icechunk repository on Cloudflare R2 (`osc-pub`).

1. Open a single virtual dataset from S3
2. Combine 5 daily files with `xr.concat`
3. Write virtual references to an Icechunk store on R2
4. Read back and plot

**Data:** [NOAA CDR NDVI v5 (AVHRR)](https://www.ncei.noaa.gov/products/climate-data-records/normalized-difference-vegetation-index)  
**Source bucket:** `s3://noaa-cdr-ndvi-pds` (anonymous access, `us-east-1`)  
**Icechunk store:** `s3://osc-pub/ndvi-cdr-icechunk` (Cloudflare R2)

In [ ]:
import os
import warnings
from dotenv import load_dotenv

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,)

## Setup: S3 store and registry

Point `obstore` at the public NDVI bucket and register it so VirtualiZarr can resolve chunk references.

In [ ]:
bucket = "s3://noaa-cdr-ndvi-pds"
base = "data/2000"

# 5 consecutive daily files — January 2000
filenames = [
    "AVHRR-Land_v005_AVH13C1_NOAA-14_20000101_c20170623095628.nc",
    "AVHRR-Land_v005_AVH13C1_NOAA-14_20000102_c20170623101557.nc",
    "AVHRR-Land_v005_AVH13C1_NOAA-14_20000103_c20170623103338.nc",
    "AVHRR-Land_v005_AVH13C1_NOAA-14_20000104_c20170623105028.nc",
    "AVHRR-Land_v005_AVH13C1_NOAA-14_20000105_c20170623110559.nc",
]

urls = [f"{bucket}/{base}/{f}" for f in filenames]

store = from_url(bucket, region="us-east-1", skip_signature=True)
registry = ObjectStoreRegistry({bucket: store})
parser = HDFParser()

urls[0]

## 1. Open a single virtual dataset

`open_virtual_dataset` reads only the file metadata — no chunk data is downloaded. The result looks like an xarray dataset but chunk references point back to the original S3 file.

We load `time`, `lat`, and `lon` as concrete arrays (they are small and needed for coordinates).

In [ ]:
vds = open_virtual_dataset(
    url=urls[0],
    parser=parser,
    registry=registry,
    loadable_variables=["time", "latitude", "longitude"],
    decode_times=True,
)
vds

In [ ]:
# ManifestArray shows the chunk layout — no data downloaded yet
vds["NDVI"].data

## 2. Combine 5 daily files

These files have a dimensionless `ncrs` coordinate (for the CRS definition) that has no index, which prevents `combine="by_coords"` from working. We use `combine="nested"` with an explicit `concat_dim="time"` instead.

In [ ]:
vds_list = [
    open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=["time", "latitude", "longitude"],
        decode_times=True,
    )
    for url in urls
]

combined_vds = xr.concat(
    vds_list,
    dim="time",
    data_vars="minimal",
    coords="minimal",
    combine_attrs="drop_conflicts",
    compat="override"
).drop_vars(["nv", "ncrs", "crs"], errors="ignore")
combined_vds

## 3. Write to Icechunk on Cloudflare R2

Create an Icechunk repository in the `osc-pub` R2 bucket and write virtual references into it. Credentials are loaded from `~/dotenv/r2-osc-pub.env`. The `VirtualChunkContainer` tells Icechunk to fetch actual chunk bytes from the original S3 files at read time — nothing is copied.

In [ ]:
load_dotenv(f'{os.environ["HOME"]}/dotenv/r2-osc-pub.env')

r2_bucket = "osc-pub"
r2_prefix = "ndvi-cdr-icechunk"
r2_endpoint = os.environ["ENDPOINT_URL"]

storage = icechunk.s3_storage(
    bucket=r2_bucket,
    prefix=r2_prefix,
    from_env=True,
    endpoint_url=r2_endpoint,
    region="auto",
    force_path_style=False,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://noaa-cdr-ndvi-pds/",
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    ),
)

repo = icechunk.Repository.open_or_create(storage, config)

session = repo.writable_session("main")
combined_vds.vz.to_icechunk(session.store)
snapshot_id = session.commit("Add 5 days NDVI CDR Jan 2000")
print("Committed:", snapshot_id)

## 4. Read back and plot

Open the Icechunk store with xarray — all 5 days appear as a single continuous dataset. Chunk data is fetched lazily from S3 on demand.

In [ ]:
credentials = icechunk.containers_credentials({
    "s3://noaa-cdr-ndvi-pds/": icechunk.s3_credentials(anonymous=True)
})

repo2 = icechunk.Repository.open(
    storage, config, authorize_virtual_chunk_access=credentials
)
session2 = repo2.readonly_session("main")

ds = xr.open_zarr(session2.store, consolidated=False, chunks=None)
ds

In [ ]:
import hvplot.xarray  # noqa

# Global NDVI map for the first day
ds["NDVI"].isel(time=0).hvplot(rasterize=True, geo=True, global_extent=True,
    x="longitude", y="latitude", tiles='OSM',
    cmap="YlGn", clim=(-0.1, 1.0),
    title="AVHRR NDVI — 2000-01-01",
    width=800, height=400,
)

In [ ]:
ds['crs'].values

In [ ]:
# Global mean NDVI over the 5-day period
ds["NDVI"].mean(["latitude", "longitude"]).hvplot(
    title="Global mean NDVI — Jan 1–5, 2000",
    ylabel="NDVI",
)